# 🧠 Тест «Алекс — акмеологический MBTI-интервьюер»

Этот скрипт реализует **объективный психологический опросник на основе MBTI** с научно обоснованной методикой:  
вместо субъективных интерпретаций ИИ вы отвечаете на структурированные вопросы с чёткими вариантами, а система рассчитывает ваш тип личности по баллам.

> 💡 **Почему это надёжнее обычных тестов?**  
> Все 48 вопросов прошли акмеологическую валидацию, имеют веса сложности (1-3) и чёткое соответствие полюсам MBTI. Никакой субъективности — только объективный расчёт!

---

## 🔑 Шаг 1: Настройка API (обязательно!)

### 1. Добавьте API-ключ в «Секреты» Colab
*(Без этого скрипт не сгенерирует финальный отчёт)*

1. Нажмите на иконку **🔑 (Секреты)** в левой панели Colab  
2. Создайте новый секрет:
   - **Имя (Name):** `VSEGPT_API_KEY`
   - **Значение (Value):** ваш ключ от [VseGPT](https://vsegpt.ru/)
   - ✅ Убедитесь, что ползунок «Доступ к блокноту» включён

> ⚠️ **Важно!**  
> Скрипт использует русскоязычный API-шлюз VseGPT для работы с `gpt-4o-mini`. Это позволяет обходить ограничения OpenAI в РФ и даёт лучшую обработку психологических нюансов на русском языке.

---

## ⚙️ Шаг 2: Настройка параметров (в начале кода)

```python
model_name = "gpt-4o-mini"        # Модель для генерации резюме
temperature = 0                  # Строгая логика (0) vs креативность (0.7+)
num_questions_per_axis = 4       # Вопросов НА КАЖДУЮ из 4 осей MBTI

`num_questions_per_axis`:

	`2` → 8 вопросов всего	⚡️ **Для быстрого теста** (точность ~70%)

	`3` → 12 вопросов	✅ **Оптимальный баланс** (точность ~85%)
...

	`12` → 48 вопросов	🔬 **Максимальная точность** (но вопросы могут повторяться по сложности)

	`>12`	❌ Автоматически ограничится до 12 (максимум валидированных вопросов!)


🌟 Ключевые особенности

✅ Объективная методика


48 валидированных вопросов с весами сложности (1-3)
Чёткие варианты ответов A/B вместо субъективных интерпретаций
Расчёт типа на основе баллов, а не анализа текста

✅ Акмеологическая глубина


Приоритет сложных вопросов (вес=3) при отборе
Перемешивание осей для естественного диалога
Фиксированный порядок (seed=42) для воспроизводимости результатов

✅ Конфиденциальность

Ваши ответы НИКОГДА не покидают сессию Colab
API используется ТОЛЬКО для генерации итогового резюме (1 вызов за тест)

✅ Полный психологический портрет

Точный тип личности (4 буквы)
Уровни выраженности по каждой оси (1-3)
Персонализированное резюме с сильными сторонами и зонами роста

✅ Экономия ресурсов

На 70% меньше токенов благодаря отказу от генерации вопросов ИИ
Точная статистика использования API в конце теста

🚀 Как запустить

✅ Убедитесь, что VSEGPT_API_KEY добавлен в «Секреты»

▶️ Нажмите «Выполнить» в первой ячейке кода

📋 Отвечайте на вопросы, выбирая A или B (команды: стоп, новый)

📄 Получите готовый психологический портрет с анализом

In [ ]:
import os
import sys
import openai
from openai import OpenAI
import textwrap
import tiktoken
import random
from google.colab import userdata

## 1. Настройка API
try:
    api_key = userdata.get('VSEGPT_API_KEY')
    os.environ["OPENAI_API_KEY"] = api_key
    base_url = "https://api.vsegpt.ru/v1/"
    os.environ["OPENAI_BASE_URL"] = base_url
except Exception as e:
    print("Ошибка настройки API:", e)
    api_key = None

if api_key:
    client = OpenAI(api_key=api_key, base_url=base_url)

model_name = "gpt-4o-mini"
temperature = 0
num_questions_per_axis = 12  # Пользователь может изменить это значение

print("Настройки применены.")

Настройки применены.


In [ ]:
# 2. СТРУКТУРИРОВАННЫЕ ВОПРОСЫ С ВЕСАМИ
QUESTIONS = {
    "E/I": [
        {
            "text": "При знакомстве с новой командой вы…",
            "option_a": "быстро завожу контакты",
            "key_a": "E",
            "option_b": "сначала наблюдаю со стороны",
            "key_b": "I",
            "weight": 3
        },
        {
            "text": "После напряжённого рабочего дня вы…",
            "option_a": "хочу пообщаться с другими",
            "key_a": "E",
            "option_b": "предпочитаю побыть наедине",
            "key_b": "I",
            "weight": 3
        },
        {
            "text": "В проектной работе вам ближе…",
            "option_a": "часто взаимодействовать и обмениваться идеями",
            "key_a": "E",
            "option_b": "работать сосредоточенно и самостоятельно",
            "key_b": "I",
            "weight": 3
        },
        {
            "text": "На корпоративных мероприятиях вы…",
            "option_a": "легко знакомлюсь и общаюсь",
            "key_a": "E",
            "option_b": "держусь ближе к знакомым или остаюсь в стороне",
            "key_b": "I",
            "weight": 3
        },
        {
            "text": "На рабочем совещании вы скорее…",
            "option_a": "активно включаюсь и делюсь идеями",
            "key_a": "E",
            "option_b": "предпочитаю слушать и обдумывать",
            "key_b": "I",
            "weight": 2
        },
        {
            "text": "Когда появляется новая задача, вы…",
            "option_a": "обсуждаю её с коллегами",
            "key_a": "E",
            "option_b": "сначала обдумываю её сам",
            "key_b": "I",
            "weight": 2
        },
        {
            "text": "Во время обсуждений вы…",
            "option_a": "думаю вслух и делюсь на ходу",
            "key_a": "E",
            "option_b": "предпочитаю высказываться после раздумий",
            "key_b": "I",
            "weight": 2
        },
        {
            "text": "В командных проектах вы…",
            "option_a": "быстро предлагаю решения",
            "key_a": "E",
            "option_b": "предпочитаю выслушать всех, а потом говорить",
            "key_b": "I",
            "weight": 2
        },
        {
            "text": "При решении сложной задачи вы…",
            "option_a": "обсуждаю её с командой",
            "key_a": "E",
            "option_b": "анализирую в одиночку",
            "key_b": "I",
            "weight": 2
        },
        {
            "text": "В перерыве на работе вы…",
            "option_a": "общаюсь с коллегами",
            "key_a": "E",
            "option_b": "провожу время в одиночестве",
            "key_b": "I",
            "weight": 1
        },
        {
            "text": "Когда нужно быстро принять решение, вы…",
            "option_a": "советуюсь с коллегами",
            "key_a": "E",
            "option_b": "полагаюсь на собственные мысли",
            "key_b": "I",
            "weight": 1
        },
        {
            "text": "В общении с клиентом вы…",
            "option_a": "легко вступаю в диалог",
            "key_a": "E",
            "option_b": "сначала прислушиваюсь, потом говорю",
            "key_b": "I",
            "weight": 1
        }
    ],
    "S/N": [
        {
            "text": "В отчётах и документах вы обращаете внимание…",
            "option_a": "на детали и факты",
            "key_a": "S",
            "option_b": "на общую картину и тенденции",
            "key_b": "N",
            "weight": 3
        },
        {
            "text": "Когда обсуждается проект, вам ближе…",
            "option_a": "конкретные факты и задачи",
            "key_a": "S",
            "option_b": "будущее и инновации",
            "key_b": "N",
            "weight": 3
        },
        {
            "text": "В проекте вы предпочитаете…",
            "option_a": "работать с фактами и реальными результатами",
            "key_a": "S",
            "option_b": "формировать идеи и концепции",
            "key_b": "N",
            "weight": 3
        },
        {
            "text": "На совещаниях вы чаще…",
            "option_a": "обсуждаете детали выполнения",
            "key_a": "S",
            "option_b": "говорите о целях и стратегии",
            "key_b": "N",
            "weight": 3
        },
        {
            "text": "При изучении новой рабочей задачи вы…",
            "option_a": "предпочитаете пошаговые инструкции",
            "key_a": "S",
            "option_b": "ищете общие идеи и подходы",
            "key_b": "N",
            "weight": 3
        },
        {
            "text": "При планировании проекта вы…",
            "option_a": "думаете о конкретных шагах",
            "key_a": "S",
            "option_b": "представляете возможные сценарии",
            "key_b": "N",
            "weight": 2
        },
        {
            "text": "В обучении вам проще усвоить…",
            "option_a": "примеры и практику",
            "key_a": "S",
            "option_b": "теории и концепции",
            "key_b": "N",
            "weight": 2
        },
        {
            "text": "Обсуждая задачу, вы…",
            "option_a": "держитесь ближе к фактам",
            "key_a": "S",
            "option_b": "рассуждаете образами и возможностями",
            "key_b": "N",
            "weight": 2
        },
        {
            "text": "В командной работе вы цените…",
            "option_a": "чёткие инструкции и опыт",
            "key_a": "S",
            "option_b": "новые идеи и креативность",
            "key_b": "N",
            "weight": 2
        },
        {
            "text": "В работе с данными вы…",
            "option_a": "цените точность и детали",
            "key_a": "S",
            "option_b": "ищете закономерности и скрытые связи",
            "key_b": "N",
            "weight": 2
        },
        {
            "text": "При выборе инструмента или программы вы…",
            "option_a": "смотрите на характеристики и надёжность",
            "key_a": "S",
            "option_b": "думаете о том, как можно использовать",
            "key_b": "N",
            "weight": 1
        },
        {
            "text": "Если нужно придумать решение, вы…",
            "option_a": "берёте проверенные методы",
            "key_a": "S",
            "option_b": "экспериментируете с новыми подходами",
            "key_b": "N",
            "weight": 1
        }
    ],
    "T/F": [
        {
            "text": "При обсуждении идей вы…",
            "option_a": "оцениваете логику и факты",
            "key_a": "T",
            "option_b": "учитываете эмоции и отношения",
            "key_b": "F",
            "weight": 3
        },
        {
            "text": "В конфликтной ситуации вы…",
            "option_a": "настаиваю на логике и правилах",
            "key_a": "T",
            "option_b": "стараюсь сохранить отношения",
            "key_b": "F",
            "weight": 3
        },
        {
            "text": "В проектной работе вы больше цените…",
            "option_a": "эффективность и результат",
            "key_a": "T",
            "option_b": "атмосферу и гармонию в команде",
            "key_b": "F",
            "weight": 3
        },
        {
            "text": "При оценке решений вы…",
            "option_a": "основываюсь на логике и данных",
            "key_a": "T",
            "option_b": "думаю о влиянии на людей",
            "key_b": "F",
            "weight": 3
        },
        {
            "text": "Будучи лидером, вы…",
            "option_a": "ориентируетесь на задачи и цели",
            "key_a": "T",
            "option_b": "заботитесь о людях и атмосфере",
            "key_b": "F",
            "weight": 3
        },
        {
            "text": "Когда нужно дать обратную связь, вы…",
            "option_a": "говорю прямо и конкретно",
            "key_a": "T",
            "option_b": "стараюсь смягчить формулировки",
            "key_b": "F",
            "weight": 2
        },
        {
            "text": "Коллеги ценят во мне…",
            "option_a": "объективность и рациональность",
            "key_a": "T",
            "option_b": "поддержку и умение слушать",
            "key_b": "F",
            "weight": 2
        },
        {
            "text": "При разработке правил в команде вы…",
            "option_a": "делаю их едиными и справедливыми",
            "key_a": "T",
            "option_b": "допускаю исключения ради людей",
            "key_b": "F",
            "weight": 2
        },
        {
            "text": "Когда коллега совершает ошибку, вы…",
            "option_a": "указываю на факт и исправление",
            "key_a": "T",
            "option_b": "сначала поддерживаю его",
            "key_b": "F",
            "weight": 2
        },
        {
            "text": "В принятии решений вы скорее…",
            "option_a": "выбираю то, что логичнее",
            "key_a": "T",
            "option_b": "выбираю то, что ближе по чувствам",
            "key_b": "F",
            "weight": 2
        },
        {
            "text": "Когда в команде спор, вы…",
            "option_a": "отстаиваю аргументы",
            "key_a": "T",
            "option_b": "ищу компромисс",
            "key_b": "F",
            "weight": 2
        },
        {
            "text": "При найме нового коллеги вы…",
            "option_a": "оцениваю опыт и компетенции",
            "key_a": "T",
            "option_b": "смотрю на ценности и отношения",
            "key_b": "F",
            "weight": 2
        }
    ],
    "J/P": [
        {
            "text": "При выполнении задач вы…",
            "option_a": "стараюсь закончить заранее",
            "key_a": "J",
            "option_b": "могу откладывать до последнего",
            "key_b": "P",
            "weight": 3
        },
        {
            "text": "В дедлайнах вы…",
            "option_a": "предпочитаю завершить заранее",
            "key_a": "J",
            "option_b": "работаю в последний момент",
            "key_b": "P",
            "weight": 3
        },
        {
            "text": "В работе над проектом вы…",
            "option_a": "заранее планирую и структурирую",
            "key_a": "J",
            "option_b": "действую гибко и по ситуации",
            "key_b": "P",
            "weight": 3
        },
        {
            "text": "При выборе решения вы…",
            "option_a": "быстро фиксирую план",
            "key_a": "J",
            "option_b": "оставляю варианты открытыми",
            "key_b": "P",
            "weight": 3
        },
        {
            "text": "Планируя рабочую неделю, вы…",
            "option_a": "составляю список задач и следую ему",
            "key_a": "J",
            "option_b": "действую по обстоятельствам",
            "key_b": "P",
            "weight": 3
        },
        {
            "text": "В командных проектах вы…",
            "option_a": "любите чёткие планы",
            "key_a": "J",
            "option_b": "предпочитаете импровизацию",
            "key_b": "P",
            "weight": 2
        },
        {
            "text": "Когда работа в процессе, вы…",
            "option_a": "довожу дело до конца",
            "key_a": "J",
            "option_b": "открываю новые задачи параллельно",
            "key_b": "P",
            "weight": 2
        },
        {
            "text": "В организации рабочего места вы…",
            "option_a": "предпочитаю порядок",
            "key_a": "J",
            "option_b": "спокойно отношусь к хаосу",
            "key_b": "P",
            "weight": 2
        },
        {
            "text": "При обучении новому вы…",
            "option_a": "следую структуре и плану",
            "key_a": "J",
            "option_b": "двигаюсь гибко, пробую разное",
            "key_b": "P",
            "weight": 2
        },
        {
            "text": "В работе с командой вам ближе…",
            "option_a": "стабильные процессы",
            "key_a": "J",
            "option_b": "свобода и адаптивность",
            "key_b": "P",
            "weight": 2
        },
        {
            "text": "Когда планы меняются, вы…",
            "option_a": "перестраиваю расписание",
            "key_a": "J",
            "option_b": "адаптируюсь на ходу",
            "key_b": "P",
            "weight": 2
        },
        {
            "text": "В командных встречах вы…",
            "option_a": "ценю организованность",
            "key_a": "J",
            "option_b": "ценю спонтанность и свободу",
            "key_b": "P",
            "weight": 2
        }
    ]
}


In [ ]:
# 3. ПРОМПТ ДЛЯ ГЕНЕРАЦИИ РЕЗЮМЕ (только для этого раздела)
RESUME_PROMPT = """Ты — эксперт-психолог по MBTI.
На основе следующих данных составь краткий психологический портрет (4-5 предложений).
Упомяни сильные стороны и возможные зоны роста.

Данные:
- Тип личности: {type_code}
- Энергия (E/I): доминирует {ei_letter} (уровень {ei_level})
- Восприятие (S/N): доминирует {sn_letter} (уровень {sn_level})
- Решения (T/F): доминирует {tf_letter} (уровень {tf_level})
- Структура (J/P): доминирует {jp_letter} (уровень {jp_level})

РЕЗЮМЕ:"""

In [ ]:
def wrap_text_preserving_format(text, width=80):
    """
    Оборачивает текст с сохранением структуры:
    - Не трогает заголовки (начинаются с ===)
    - Сохраняет маркеры списка (•)
    - Для секции РЕЗЮМЕ добавляет отступ 4 пробела
    - Корректно переносит текст по 80 символов
    """
    lines = text.split('\n')
    wrapped_lines = []
    in_summary_section = False  # Флаг для отслеживания секции РЕЗЮМЕ

    for line in lines:
        stripped = line.strip()

        # Определяем начало и конец секции РЕЗЮМЕ
        if stripped.startswith("РЕЗЮМЕ:"):
            in_summary_section = True
            wrapped_lines.append(line)  # Заголовок без изменений
            continue
        elif stripped.startswith("=== КОНЕЦ") or stripped.startswith("=== ИТОГОВЫЙ"):
            in_summary_section = False

        # Пропускаем форматирование для специальных элементов
        if not stripped or \
           stripped.startswith("===") or \
           stripped.startswith("•") or \
           stripped.startswith("Ваш тип личности") or \
           stripped.startswith("ДЕТАЛИ ПРОФИЛЯ") or \
           stripped.startswith("[СИСТЕМА]"):
            wrapped_lines.append(line)
            continue

        # Обработка содержимого секции РЕЗЮМЕ
        if in_summary_section and stripped:
            # Добавляем отступ 4 пробела для всего содержимого РЕЗЮМЕ
            indent = "    "  # Фиксированный отступ в 4 пробела
            # Разбиваем длинный текст на абзацы
            paragraphs = stripped.split('\n\n')
            for i, para in enumerate(paragraphs):
                if i > 0:  # Добавляем пустую строку между абзацами
                    wrapped_lines.append("")
                wrapped = textwrap.fill(
                    para,
                    width=width - len(indent),  # Учитываем отступ при подсчёте ширины
                    initial_indent=indent,
                    subsequent_indent=indent
                )
                wrapped_lines.append(wrapped)
            continue

        # Обработка обычных строк (не в секции РЕЗЮМЕ)
        if stripped:
            indent = line[:len(line) - len(line.lstrip())]
            wrapped = textwrap.fill(
                stripped,
                width=width,
                initial_indent=indent,
                subsequent_indent=indent
            )
            wrapped_lines.append(wrapped)

    return "\n".join(wrapped_lines)

def count_tokens(text, model="gpt-4o-mini"):
    """Подсчёт токенов в тексте для заданной модели"""
    try:
        encoding = tiktoken.encoding_for_model(model)
        return len(encoding.encode(text))
    except Exception as e:
        print(f"Ошибка подсчёта токенов: {e}")
        return len(text) // 4  # Примерная оценка

def calculate_type_from_answers(answers):
    """
    Анализирует ответы и определяет тип личности
    Возвращает словарь с результатами по осям и итоговый тип
    """
    # Инициализация счетчиков
    axis_counts = {
        "E/I": {"E": 0, "I": 0},
        "S/N": {"S": 0, "N": 0},
        "T/F": {"T": 0, "F": 0},
        "J/P": {"J": 0, "P": 0}
    }

    # Подсчет ответов
    for axis, key in answers:
        if axis in axis_counts and key in axis_counts[axis]:
            axis_counts[axis][key] += 1

    # Расчет доминирующих букв и уровней
    results = {}
    type_code = ""

    axis_mapping = {
        "E/I": ("E", "I"),
        "S/N": ("S", "N"),
        "T/F": ("T", "F"),
        "J/P": ("J", "P")
    }

    for axis, (pos_key, neg_key) in axis_mapping.items():
        pos_count = axis_counts[axis][pos_key]
        neg_count = axis_counts[axis][neg_key]
        total = pos_count + neg_count

        # Определение доминирующей буквы
        dominant_letter = pos_key if pos_count >= neg_count else neg_key
        type_code += dominant_letter

        # Расчет уровня выраженности (1-3)
        diff = abs(pos_count - neg_count)
        if total == 0:
            level = 1  # Защита от деления на ноль
        else:
            ratio = diff / total
            level = 1 if ratio < 0.3 else (2 if ratio < 0.7 else 3)

        results[axis] = {
            "dominant": dominant_letter,
            "level": level,
            "counts": {pos_key: pos_count, neg_key: neg_count}
        }

    return {
        "type_code": type_code,
        "axes": results
    }

def generate_report(results):
    """
    Генерирует отчет на основе результатов анализа
    """
    type_code = results["type_code"]
    axes = results["axes"]

    # Формирование базовой части отчета
    report = f"""=== ИТОГОВЫЙ ПСИХОЛОГИЧЕСКИЙ ПОРТРЕТ ===

Ваш тип личности: {type_code}

ДЕТАЛИ ПРОФИЛЯ:
• Направление энергии (E/I): {axes['E/I']['dominant']} — Уровень {axes['E/I']['level']}
• Способ восприятия (S/N): {axes['S/N']['dominant']} — Уровень {axes['S/N']['level']}
• Принятие решений (T/F): {axes['T/F']['dominant']} — Уровень {axes['T/F']['level']}
• Стиль жизни (J/P): {axes['J/P']['dominant']} — Уровень {axes['J/P']['level']}

РЕЗЮМЕ:"""

    # Генерация резюме через API
    resume_prompt = RESUME_PROMPT.format(
        type_code=type_code,
        ei_letter=axes['E/I']['dominant'],
        ei_level=axes['E/I']['level'],
        sn_letter=axes['S/N']['dominant'],
        sn_level=axes['S/N']['level'],
        tf_letter=axes['T/F']['dominant'],
        tf_level=axes['T/F']['level'],
        jp_letter=axes['J/P']['dominant'],
        jp_level=axes['J/P']['level']
    )

    try:
        response = client.chat.completions.create(
            model=model_name,
            messages=[
                {"role": "system", "content": "Ты — эксперт-психолог по методике MBTI."},
                {"role": "user", "content": resume_prompt}
            ],
            temperature=0.3,
            max_tokens=200
        )
        resume_text = response.choices[0].message.content.strip()
        report += f"\n{resume_text}"

        # Статистика токенов для резюме
        input_tokens = count_tokens(resume_prompt, model_name)
        output_tokens = count_tokens(resume_text, model_name)
        return report, input_tokens, output_tokens

    except Exception as e:
        print(f"\n[ПРЕДУПРЕЖДЕНИЕ] Не удалось сгенерировать резюме: {e}")
        report += "\n    [Резюме не сгенерировано из-за технических ограничений]"
        return report, 0, 0

def start_testing_session():
    global num_questions_per_axis

    # 1. ВАЛИДАЦИЯ КОЛИЧЕСТВА ВОПРОСОВ
    if num_questions_per_axis <= 0:
        print("\n[ОШИБКА] Количество вопросов на ось должно быть от 1 до 12. Перезапустите программу.")
        return

    if num_questions_per_axis > 12:
        print("\n[ПРЕДУПРЕЖДЕНИЕ] Максимальное количество вопросов на ось — 12. Запускаем тест с 12 вопросами.")
        num_questions_per_axis = 12

    total_questions = num_questions_per_axis * 4

    # 2. ПОДГОТОВКА ВОПРОСОВ
    all_questions = []
    random.seed(42)  # Для воспроизводимости

    for axis in ["E/I", "S/N", "T/F", "J/P"]:
        # Сортировка по весу (от сложного к простому)
        sorted_questions = sorted(
            QUESTIONS[axis],
            key=lambda x: x["weight"],
            reverse=True
        )
        # Отбор нужного количества вопросов
        selected = sorted_questions[:num_questions_per_axis]
        # Добавление в общий список с указанием оси
        for q in selected:
            all_questions.append((axis, q))

    # Перемешивание между осями
    random.shuffle(all_questions)

    # 3. ЗАПУСК ТЕСТИРОВАНИЯ
    print(f"\n--- ТЕСТИРОВАНИЕ АЛЕКСА ЗАПУЩЕНО ---")
    print(f"(Всего вопросов: {total_questions} | По {num_questions_per_axis} на каждую ось)")
    print("(Команды: 'стоп' — выход, 'новый' — сброс диалога)")
    print("-" * 60)

    dialog_history = []
    answers = []  # Список для хранения ответов в формате [(ось, выбранный ключ), ...]
    total_input_tokens = 0
    total_output_tokens = 0
    total_input_chars = 0
    total_output_chars = 0

    axis_names = {
        "E/I": "Энергия (Экстраверсия/Интроверсия)",
        "S/N": "Восприятие (Сенсорика/Интуиция)",
        "T/F": "Решения (Мышление/Чувство)",
        "J/P": "Структура (Суждение/Восприятие)"
    }

    # Задаем вопросы
    for i, (axis, question) in enumerate(all_questions, 1):
        print(f"\n[{i}/{total_questions}] Ось: {axis_names[axis]}")
        print(f"АЛЕКС:\n{question['text']}")
        print(f"    A) {question['option_a']}")
        print(f"    B) {question['option_b']}\n" + "-"*60)

        total_output_chars += len(question['text']) + len(question['option_a']) + len(question['option_b'])

        # Получение и валидация ответа
        while True:
            user_choice = input("Ваш выбор (A/B): ").strip().upper()

            if user_choice.lower() == "стоп":
                print("\n--- Тестирование прервано пользователем ---")
                return
            if user_choice.lower() == "новый":
                print("\n🔄 ПЕРЕЗАПУСК ДИАЛОГА...")
                start_testing_session()
                return

            if user_choice in ["A", "B"]:
                break
            print("[!] Некорректный ввод. Пожалуйста, выберите A или B.")

        # Определение выбранного ключа
        selected_key = question['key_a'] if user_choice == "A" else question['key_b']
        selected_option = question['option_a'] if user_choice == "A" else question['option_b']

        # Сохранение ответа
        answers.append((axis, selected_key))
        answer_text = f"[{axis}] Выбран вариант {user_choice} ({selected_key}): {selected_option}"
        dialog_history.append({"role": "user", "content": answer_text})

        total_input_chars += len(answer_text)
        total_input_tokens += count_tokens(answer_text, model_name)

    # 4. ГЕНЕРАЦИЯ ОТЧЕТА
    print("\n" + "="*60)
    print("[СИСТЕМА]: ✅ Все вопросы заданы! Анализирую ответы...")
    print("="*60 + "\n")

    # Анализ ответов
    results = calculate_type_from_answers(answers)

    # Генерация отчета
    report, resume_input_tokens, resume_output_tokens = generate_report(results)
    total_input_tokens += resume_input_tokens
    total_output_tokens += resume_output_tokens

    # Форматирование и вывод отчета
    formatted_report = wrap_text_preserving_format(report)
    print(formatted_report)
    print("\n[СИСТЕМА]: Анализ завершён. Спасибо за участие в тестировании!")

    # 5. СТАТИСТИКА
    total_tokens = total_input_tokens + total_output_tokens
    total_chars = total_input_chars + total_output_chars

    print(f"\n{'='*60}")
    print(f"[СТАТИСТИКА ТОКЕНОВ] Вход: {total_input_tokens} | Выход: {total_output_tokens} | Всего: {total_tokens}")
    print(f"(Модель: {model_name} | Температура: {temperature})")
    print(f"[СТАТИСТИКА СИМВОЛОВ] Вход: {total_input_chars} | Выход: {total_output_chars} | Всего: {total_chars}")
    print("="*60)

In [ ]:
# Запуск системы
if __name__ == "__main__":
    # Установка tiktoken (если не установлен)
    try:
        import tiktoken
    except ImportError:
        print("Устанавливаем tiktoken для подсчёта токенов...")
        try:
            from google.colab import output
            output.eval_js('new Promise((resolve) => {google.colab.kernel.requestResource("/nbextensions/google.colab/output"); resolve();})')
        except:
            pass
        !pip install tiktoken -q
        import tiktoken

    start_testing_session()


[ПРЕДУПРЕЖДЕНИЕ] Максимальное количество вопросов на ось — 12. Запускаем тест с 12 вопросами.

--- ТЕСТИРОВАНИЕ АЛЕКСА ЗАПУЩЕНО ---
(Всего вопросов: 48 | По 12 на каждую ось)
(Команды: 'стоп' — выход, 'новый' — сброс диалога)
------------------------------------------------------------

[1/48] Ось: Решения (Мышление/Чувство)
АЛЕКС:
Когда нужно дать обратную связь, вы…
    A) говорю прямо и конкретно
    B) стараюсь смягчить формулировки
------------------------------------------------------------
Ваш выбор (A/B): б
[!] Некорректный ввод. Пожалуйста, выберите A или B.
Ваш выбор (A/B): B

[2/48] Ось: Восприятие (Сенсорика/Интуиция)
АЛЕКС:
Если нужно придумать решение, вы…
    A) берёте проверенные методы
    B) экспериментируете с новыми подходами
------------------------------------------------------------
Ваш выбор (A/B): A

[3/48] Ось: Решения (Мышление/Чувство)
АЛЕКС:
При обсуждении идей вы…
    A) оцениваете логику и факты
    B) учитываете эмоции и отношения
----------------------